# Vectorless RAG — PageIndex + Anthropic Claude

> **Reasoning-based document retrieval without a vector database.**  
> Instead of chunking & embedding, PageIndex builds a **hierarchical tree index** over your PDF.  
> Claude then *reasons* its way to the right section — like a human expert scanning a table of contents.

---

### Pipeline at a glance

```
PDF  →  [PageIndex] Tree Index  →  LLM-guided Tree Search  →  [Claude] Final Answer
```

| Stage | Tool | Runs |
|-------|------|------|
| Indexing | PageIndex | Once per document |
| Retrieval | PageIndex Chat API | Per query |
| Generation | Anthropic Claude | Per query |

**Benchmark:** 98.7% accuracy on FinanceBench — outperforming vector-based RAG on complex financial documents.

---

### Notebook Structure

| Step | Description |
|------|-------------|
| **0** | Install dependencies & configure API keys |
| **1** | Submit a PDF to PageIndex and build the tree index |
| **2** | Inspect the hierarchical document tree |
| **3** | Core RAG — retrieve context + generate answer with Claude |
| **4** | Multi-document cross-synthesis |
| **5** | Vision RAG — analyse charts, tables, and scanned pages |

---
## Step 0 — Install Dependencies & Configure API Keys

In [ ]:
# Install all required packages
# pageindex  — hierarchical tree indexing + retrieval
# anthropic  — Claude SDK for answer generation
# pymupdf    — render PDF pages as images (Vision RAG)
# python-dotenv — load .env file for API keys

%pip install -q pageindex anthropic pymupdf python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from a .env file if present
# The .env file should contain:
#   PAGEINDEX_API_KEY=pi-...
#   ANTHROPIC_API_KEY=sk-ant-...
load_dotenv()

# Alternatively, set keys directly here (not recommended for shared notebooks)
# os.environ["PAGEINDEX_API_KEY"] = "pi-..."
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

# Verify keys are loaded
assert os.environ.get("PAGEINDEX_API_KEY"), "Missing PAGEINDEX_API_KEY"
assert os.environ.get("ANTHROPIC_API_KEY"), "Missing ANTHROPIC_API_KEY"

print("✅ API keys loaded successfully")

In [ ]:
# Initialise clients — used throughout the notebook
import anthropic
from pageindex import PageIndexClient

pi_client        = PageIndexClient(api_key=os.environ["PAGEINDEX_API_KEY"])
anthropic_client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

print("✅ PageIndex client ready")
print("✅ Anthropic Claude client ready")

---
## Step 1 — Submit a PDF & Build the Tree Index

PageIndex processes your PDF and builds a **hierarchical tree index** — a machine-readable table of contents where every node knows its:
- `title` — section heading
- `page_index` — page number(s) it spans
- `nodes` — child sections (recursive)

This replaces the traditional chunk-and-embed step entirely.  
You do this **once per document**. The returned `doc_id` is reused for all subsequent queries.

> ⏱ Indexing typically completes in 30–120 seconds depending on document length.

In [ ]:
import time

def submit_and_wait(pdf_path: str) -> str:
    """
    Submit a PDF to PageIndex and poll until the tree index is built.

    Args:
        pdf_path: Local path to the PDF file.

    Returns:
        doc_id: Unique PageIndex document identifier. Save this for reuse.

    Raises:
        RuntimeError: If PageIndex reports a processing failure.
    """
    # Submit the document — PageIndex begins building the tree index
    result = pi_client.submit_document(pdf_path)
    doc_id = result["doc_id"]
    print(f"📤 Submitted: {doc_id}")
    print("⏳ Waiting for indexing to complete...")

    # Poll every 5 seconds until status is 'completed' or 'failed'
    while True:
        status = pi_client.get_document(doc_id)["status"]
        if status == "completed":
            print(f"✅ Ready: {doc_id}")
            return doc_id
        if status == "failed":
            raise RuntimeError("PageIndex processing failed. Check the PDF and retry.")
        print(f"   still processing (status: {status})...")
        time.sleep(5)

In [ ]:
# ── Configure: set the path to your PDF ──────────────────────────────────────
PDF_PATH = "./annual_report.pdf"   # Change to your PDF path
# ─────────────────────────────────────────────────────────────────────────────

doc_id = submit_and_wait(PDF_PATH)

print(f"""
╔══════════════════════════════════════════╗
  Save your doc_id — reuse it across sessions:
  {doc_id}
╚══════════════════════════════════════════╝
""")

In [ ]:
# ── If you already indexed the PDF previously, paste the doc_id here ─────────
# doc_id = "pi-doc-xxxxxxxx"   # Uncomment and fill in to skip re-indexing
# ─────────────────────────────────────────────────────────────────────────────

---
## Step 2 — Inspect the Document Tree

This is something you **cannot** do with vector RAG: see exactly what structure was built.  
The tree gives you full transparency into how PageIndex has organised your document — no black boxes, no cosine distances.

Each node in the tree corresponds to a logical section of the document.

In [ ]:
def print_tree(nodes: list, depth: int = 0) -> None:
    """
    Recursively print the document tree with indentation.

    Args:
        nodes: List of tree nodes from PageIndex.
        depth: Current recursion depth (controls indentation).

    Example output:
        • Financial Highlights (pp.3)
          • Revenue Summary (pp.4)
          • Cost of Goods Sold (pp.5)
        • Risk Factors (pp.12)
          • Market Risk (pp.13)
    """
    for node in nodes:
        indent = "  " * depth
        pages  = f"pp.{node['page_index']}"
        print(f"{indent}• {node['title']} ({pages})")
        # Recurse into child sections
        if node.get("nodes"):
            print_tree(node["nodes"], depth + 1)


def inspect_tree(doc_id: str) -> list:
    """
    Fetch the hierarchical tree index from PageIndex and display it.

    Args:
        doc_id: PageIndex document identifier from Step 1.

    Returns:
        tree: Raw tree data (list of nodes) for programmatic use.
    """
    tree = pi_client.get_tree(doc_id)["result"]
    print(f"Document tree for: {doc_id}")
    print("-" * 50)
    print_tree(tree)
    return tree

In [ ]:
tree = inspect_tree(doc_id)

# You can also inspect the raw structure programmatically
print(f"\n📊 Total top-level sections: {len(tree)}")

---
## Step 3 — Core RAG Pipeline: Retrieve + Generate

This is the main pipeline:

1. **Retrieve** — PageIndex's Chat API walks the tree, reasons about which sections contain the answer, and returns relevant text with page citations.
2. **Generate** — Claude receives the retrieved context and produces the final answer.

These two responsibilities are kept separate — giving you full control over both retrieval quality and generation style.

> **Note:** The original project used LangChain + OpenAI GPT-4o. This notebook uses the **Anthropic SDK + Claude** directly, matching the architecture shown in the Substack post.

In [ ]:
def retrieve_context(doc_id: str, query: str) -> str:
    """
    Use PageIndex's LLM-guided tree search to retrieve relevant sections.

    PageIndex walks the document tree, reasons at each node about whether
    the answer is in that section or its children, and returns the relevant
    text with page references — no vector similarity required.

    Args:
        doc_id: PageIndex document identifier.
        query:  Natural language question.

    Returns:
        context: Retrieved text with page references.
    """
    response = pi_client.chat_completions(
        messages=[{
            "role": "user",
            "content": (
                f"Find sections relevant to: {query}\n"
                f"Return only the retrieved text with page references."
            )
        }],
        doc_id=doc_id
    )
    return response["choices"][0]["message"]["content"]


def generate_answer(query: str, context: str) -> str:
    """
    Use Claude to synthesise a precise answer from the retrieved context.

    Claude is instructed to:
    - Use ONLY the provided context (no hallucination)
    - Always cite page numbers

    Args:
        query:   The original question.
        context: Retrieved text from PageIndex (with page references).

    Returns:
        answer: Claude's synthesised response.
    """
    resp = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system="You are a precise document analyst. Use ONLY the provided context. Always cite page numbers in your answer.",
        messages=[{
            "role": "user",
            "content": f"## Context\n\n{context}\n\n## Question\n\n{query}"
        }]
    )
    return resp.content[0].text


def run_pipeline(doc_id: str, query: str) -> str:
    """
    Full retrieve + generate pipeline.

    Args:
        doc_id: PageIndex document identifier.
        query:  Natural language question.

    Returns:
        answer: Final answer from Claude with page citations.
    """
    print(f"🔍 Query: {query}\n")

    print("📚 Retrieving context from PageIndex tree...")
    context = retrieve_context(doc_id, query)
    print(f"   Retrieved {len(context)} characters of context\n")

    print("🤖 Generating answer with Claude...\n")
    return generate_answer(query, context)

In [ ]:
# ── Run a query against your indexed document ─────────────────────────────────
query = "What was Q3 net revenue and how did it compare to Q2?"
# ─────────────────────────────────────────────────────────────────────────────

answer = run_pipeline(doc_id, query)

print("Answer:")
print("-" * 50)
print(answer)

In [ ]:
# ── Streaming variant — useful for interactive apps ───────────────────────────
# Instead of waiting for the full response, stream tokens as they arrive.

def stream_answer(query: str, context: str) -> None:
    """
    Stream Claude's answer token-by-token to stdout.

    Args:
        query:   The original question.
        context: Retrieved text from PageIndex.
    """
    print("Streaming answer: ", end="", flush=True)
    with anthropic_client.messages.stream(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        system="You are a precise document analyst. Use ONLY the provided context. Cite page numbers.",
        messages=[{
            "role": "user",
            "content": f"## Context\n\n{context}\n\n## Question\n\n{query}"
        }]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
    print()  # newline after stream ends


# Example usage (uncomment to run):
# stream_query = "What are the key risk factors mentioned?"
# stream_context = retrieve_context(doc_id, stream_query)
# stream_answer(stream_query, stream_context)

---
## Step 4 — Multi-Document Cross-Synthesis

Pass a **list** of `doc_id`s to query multiple documents simultaneously.  
PageIndex retrieves relevant sections from all documents and tags each by source.  
Claude then synthesises a comparative answer, noting agreements and contradictions.

**Ideal for:**
- Comparing year-over-year filings (e.g., FY2023 vs FY2024 annual reports)
- Reviewing multiple contract versions
- Cross-referencing research papers

> You need to have indexed at least **two** documents (Steps 1–2) before running this step.

In [ ]:
def retrieve_multi_context(doc_ids: list[str], query: str) -> str:
    """
    Retrieve tagged sections from multiple documents via PageIndex.

    PageIndex queries all documents simultaneously and tags each retrieved
    section with its source document, enabling cross-document comparison.

    Args:
        doc_ids: List of PageIndex document identifiers (min 2).
        query:   Natural language question to answer across all documents.

    Returns:
        context: Tagged sections from all documents with source labels.
    """
    retrieval = pi_client.chat_completions(
        messages=[{
            "role": "user",
            "content": f"Find and tag sections from ALL documents relevant to: {query}"
        }],
        doc_id=doc_ids   # Pass a list — not a string — for multi-doc retrieval
    )
    return retrieval["choices"][0]["message"]["content"]


def multi_doc_query(doc_ids: list[str], query: str) -> str:
    """
    Full multi-document retrieve + synthesise pipeline.

    Args:
        doc_ids: List of PageIndex document identifiers (min 2).
        query:   Comparative or cross-document question.

    Returns:
        answer: Claude's synthesised cross-document answer with source citations.
    """
    if len(doc_ids) < 2:
        raise ValueError("Provide at least two doc_ids for multi-document synthesis.")

    print(f"📚 Retrieving from {len(doc_ids)} documents: {doc_ids}")
    context = retrieve_multi_context(doc_ids, query)

    print("🤖 Generating cross-document answer with Claude...\n")
    resp = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=(
            "You are a cross-document analyst. "
            "Compare information across the provided sources. "
            "Note agreements and contradictions. "
            "Always cite document names and page numbers."
        ),
        messages=[{
            "role": "user",
            "content": f"## Multi-document context\n\n{context}\n\n## Question\n\n{query}"
        }]
    )
    return resp.content[0].text

In [ ]:
# ── Configure: add your second doc_id ────────────────────────────────────────
# Index a second PDF first (e.g., last year's annual report) and paste its
# doc_id below. Then uncomment the block to run multi-document synthesis.

# doc_id_2 = "pi-doc-xxxxxxxx"   # Second document's doc_id
#
# multi_query = "How did operating margins change year over year?"
#
# answer = multi_doc_query([doc_id, doc_id_2], multi_query)
# print("Answer:")
# print("-" * 50)
# print(answer)
# ─────────────────────────────────────────────────────────────────────────────

print("ℹ️  Uncomment the block above once you have a second doc_id ready.")

---
## Step 5 — Vision RAG: Charts, Tables & Scanned Pages

Charts, diagrams, and scanned pages don't exist in the text layer — traditional RAG simply cannot reach them.  
The Vision RAG pipeline handles this in three steps:

1. **Identify** — PageIndex tree search finds the relevant page numbers.
2. **Render** — PyMuPDF renders those pages as base64-encoded PNG images at 150 DPI.
3. **Analyse** — Claude's vision capability reads the images and answers with specific values.

> ⚠️ **Important:** PageIndex `page_index` is **1-based**. PyMuPDF `load_page()` is **0-based**.  
> The `pg - 1` conversion in `render_pages()` handles this. Easy to miss, hard to debug.

In [ ]:
import re
import base64
import fitz  # PyMuPDF


def find_relevant_pages(doc_id: str, query: str) -> list[int]:
    """
    Use the PageIndex tree to identify which page numbers are relevant to the query.

    Strategy:
    1. Collect all valid page numbers from the tree (the 'pool').
    2. Ask PageIndex which pages are relevant — returns a text response.
    3. Parse page numbers from the response and filter against the pool.
    4. Fall back to the first 3 pages if nothing is found.

    Args:
        doc_id: PageIndex document identifier.
        query:  Natural language question (ideally about visual content).

    Returns:
        pages: Sorted list of relevant 1-based page numbers (max 5).
    """
    tree = pi_client.get_tree(doc_id)["result"]

    def all_pages(nodes: list) -> list[int]:
        """Recursively collect all page numbers from the tree."""
        nums = []
        for n in nodes:
            if "page_index" in n:
                nums.append(n["page_index"])
            if n.get("nodes"):
                nums.extend(all_pages(n["nodes"]))
        return nums

    # Ask PageIndex which pages are relevant
    resp = pi_client.chat_completions(
        messages=[{
            "role": "user",
            "content": (
                f"Which pages contain content relevant to: {query}\n"
                f"Return page numbers only."
            )
        }],
        doc_id=doc_id
    )
    text = resp["choices"][0]["message"]["content"]

    # Filter returned numbers against valid pages in the tree
    pool  = set(all_pages(tree))
    found = [int(n) for n in re.findall(r"\b(\d+)\b", text) if int(n) in pool]

    # Return up to 5 pages; fall back to first 3 if nothing found
    return sorted(set(found))[:5] or sorted(pool)[:3]


def render_pages(pdf_path: str, pages: list[int]) -> list[dict]:
    """
    Render PDF pages to base64-encoded PNG images at 150 DPI.

    150 DPI provides good quality for charts and tables while keeping
    image sizes manageable for the Claude API.

    Args:
        pdf_path: Local path to the PDF file.
        pages:    1-based page numbers to render.

    Returns:
        List of dicts with keys:
            'page' (int): 1-based page number
            'b64'  (str): base64-encoded PNG image

    Note:
        PageIndex page_index is 1-based; PyMuPDF load_page() is 0-based.
        The `pg - 1` conversion handles this offset.
    """
    doc    = fitz.open(pdf_path)
    mat    = fitz.Matrix(150 / 72, 150 / 72)  # Scale factor for 150 DPI
    result = []

    for pg in pages:
        # pg is 1-based (PageIndex); load_page() expects 0-based (PyMuPDF)
        pix = doc.load_page(pg - 1).get_pixmap(matrix=mat, alpha=False)
        result.append({
            "page": pg,
            "b64":  base64.standard_b64encode(pix.tobytes("png")).decode()
        })

    doc.close()
    return result


def vision_answer(query: str, page_images: list[dict]) -> str:
    """
    Send rendered page images to Claude for visual analysis.

    Builds a multimodal message interleaving page labels and PNG images,
    then appends the question. Claude reads all charts, tables, and diagrams
    visually and answers with specific values and page citations.

    Args:
        query:       Natural language question about visual content.
        page_images: List of rendered pages from render_pages().

    Returns:
        answer: Claude's visual analysis with specific values and page citations.
    """
    # Build interleaved text + image content blocks
    content = []
    for img in page_images:
        # Label each page
        content.append({
            "type": "text",
            "text": f"\n### Page {img['page']}\n"
        })
        # Attach the rendered PNG image
        content.append({
            "type": "image",
            "source": {
                "type":       "base64",
                "media_type": "image/png",
                "data":       img["b64"]
            }
        })

    # Append the question at the end
    content.append({"type": "text", "text": f"\n## Question\n\n{query}"})

    resp = anthropic_client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=2048,
        system=(
            "Read all charts, tables, and diagrams carefully. "
            "Answer precisely with specific values. Cite page numbers."
        ),
        messages=[{"role": "user", "content": content}]
    )
    return resp.content[0].text


def run_vision_pipeline(doc_id: str, pdf_path: str, query: str) -> str:
    """
    Full Vision RAG pipeline: identify pages → render → analyse.

    Args:
        doc_id:   PageIndex document identifier.
        pdf_path: Local path to the PDF file.
        query:    Question about charts, tables, or scanned content.

    Returns:
        answer: Claude's visual analysis answer.
    """
    print(f"🔍 Query: {query}\n")

    print("🌳 Finding relevant pages via PageIndex tree...")
    pages = find_relevant_pages(doc_id, query)
    print(f"   Relevant pages: {pages}\n")

    print("🖼️  Rendering pages to images (150 DPI)...")
    images = render_pages(pdf_path, pages)
    print(f"   Rendered {len(images)} page(s)\n")

    print("🤖 Sending to Claude vision...\n")
    return vision_answer(query, images)

In [ ]:
# ── Run a vision query against your PDF ──────────────────────────────────────
vision_query = "What does the revenue breakdown chart show for FY2024?"
# ─────────────────────────────────────────────────────────────────────────────

vision_result = run_vision_pipeline(doc_id, PDF_PATH, vision_query)

print("Answer:")
print("-" * 50)
print(vision_result)

---
## Summary

You now have a complete **Vectorless RAG** pipeline using **PageIndex + Anthropic Claude**:

| Step | What it does | Key function |
|------|-------------|----------------|
| 1 | Index PDF → hierarchical tree | `submit_and_wait()` |
| 2 | Inspect tree structure | `inspect_tree()` |
| 3 | Retrieve context + generate answer | `run_pipeline()` |
| 4 | Query across multiple documents | `multi_doc_query()` |
| 5 | Analyse charts/tables visually | `run_vision_pipeline()` |

### Why this outperforms vector RAG on structured documents

| | Vector RAG | Vectorless RAG |
|--|-----------|----------------|
| **Retrieval method** | Approximate similarity (ANN) | LLM-guided tree reasoning |
| **Context preserved** | ✗ Chunks break boundaries | ✓ Natural sections intact |
| **Explainability** | Cosine scores | Page numbers + section titles |
| **Charts & diagrams** | ✗ Not accessible | ✓ Vision RAG pipeline |
| **Re-index cost** | Hours (re-embed corpus) | Seconds |
| **Infrastructure** | GPU + vector store | CPU only |

### Useful links
- [PageIndex docs](https://pageindex.ai)
- [Anthropic API docs](https://docs.anthropic.com)
- [Claude model reference](https://docs.anthropic.com/en/docs/about-claude/models)